# 第2章课堂版：SFT 监督微调（教学版）

这章回答一个问题：**文本是怎么变成模型可学习的数字信息的**。

你会看到：
- 文本 -> 分词 -> 向量
- 向量 -> 类别概率
- 概率 -> loss -> 参数更新


## 学习目标

- 理解 `tokenize / featurize / softmax / loss` 的角色。
- 能解释“为什么平均 loss 会下降”。
- 能通过改训练样本影响预测结果。


In [ ]:
from pathlib import Path
import subprocess, sys, re

ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    if (ROOT.parent / 'projects').exists():
        ROOT = ROOT.parent
    else:
        raise RuntimeError(f'未找到项目根目录（应包含 projects/ 目录），当前目录: {Path.cwd()}')

print('使用项目目录:', ROOT)

def run_py(rel_path: str) -> str:
    # 执行项目脚本并返回标准输出文本
    result = subprocess.run(
        [sys.executable, str(ROOT / rel_path)],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=True,
    )
    print(result.stdout)
    return result.stdout


## 第一步：运行 SFT 脚本


In [ ]:
out = run_py('projects/project-01-sft/train.py')


## 第二步：抓取关键指标


In [ ]:
loss_values = [float(x) for x in re.findall(r'平均loss=([0-9.]+)', out)]
print('loss 序列:', loss_values)
print('是否整体下降:', loss_values[-1] < loss_values[0])

pred_lines = [line for line in out.splitlines() if line.strip().startswith('-> 类别=')]
print('\n预测摘要:')
for line in pred_lines:
    print(line.strip())


## 第三步：白话解释每个模块

- `tokenize`：把句子拆词。
- `featurize`：把词变成数字向量（词袋）。
- `softmax`：把分数变成概率。
- `loss`：当前预测错得多严重。
- 参数更新：让下次同类输入更容易被分到正确类别。


## 动手练习

1. 在 `projects/project-01-sft/config.yaml` 把 `learning_rate` 改成 `0.1`。
2. 重新运行本 Notebook 的“运行脚本”单元。
3. 对比 loss 下降速度。

进阶：
- 在 `DATASET` 增加一条缓存类样本（标签 1），观察预测是否更稳定。
